# Data Munging with Python
**Complete End-to-End Analysis — Beginner Friendly**

This notebook is a Python translation of the R tidyverse data munging script.
We use: `numpy`, `pandas`, `scikit-learn`, `matplotlib`, and `seaborn`.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Make plots look nice
sns.set_theme(style='whitegrid')
print('All libraries imported!')

## 2. Create the Messy Employee Dataset
This is our raw, messy data — just like real-world data.

In [ ]:
employees = pd.DataFrame({
    'employee_id': [101, 102, 103, 104, 105, 103, 106, 107],

    'name': [
        'Ali Khan',
        ' Sara Ahmed ',   # extra spaces
        'Ahmed Raza',
        'Fatima Noor',
        'Usman Ali',
        'Ahmed Raza',     # duplicate!
        'Ayesha Malik',
        'Bilal Shah'
    ],

    'age': [25, np.nan, 32, 29, 41, 32, 27, 35],  # np.nan = missing

    'gender': ['M', 'Female', 'Male', 'F', 'male', 'Male', 'F', 'MALE'],  # inconsistent!

    'salary': ['50000', '$60,000', '55000', None, '85000', '55000', '65000', '$72,000'],  # messy strings!

    'department': ['IT', 'HR', 'it', 'Finance', 'HR ', 'it', 'FINANCE', ' Hr']  # inconsistent cases!
})

employees

## 3. Inspect the Raw Dataset

In [ ]:
# First 5 rows
print('--- First 5 rows ---')
print(employees.head())

print('\n--- Last 5 rows ---')
print(employees.tail())

print('\n--- Shape (rows, columns) ---')
print(employees.shape)

print('\n--- Column names ---')
print(employees.columns.tolist())

print('\n--- Data types (like str() in R) ---')
print(employees.dtypes)

print('\n--- Summary statistics ---')
print(employees.describe(include='all'))

## 4. Identify Missing Values

In [ ]:
# True/False table of missing values
print('--- Missing value map ---')
print(employees.isnull())

print('\n--- Total missing values ---')
print(employees.isnull().sum().sum())

print('\n--- Missing values per column ---')
print(employees.isnull().sum())

print('\n--- Rows that have at least one missing value ---')
print(employees[employees.isnull().any(axis=1)])

## 5. Identify Duplicate Records

In [ ]:
# Check which rows are duplicates
print('--- Duplicate row flags ---')
print(employees.duplicated())

print('\n--- Number of duplicate rows ---')
print(employees.duplicated().sum())

print('\n--- Show duplicate rows ---')
print(employees[employees.duplicated()])

print('\n--- Duplicate employee_ids ---')
dup_ids = employees['employee_id'].value_counts()
print(dup_ids[dup_ids > 1])

## 6. Remove Duplicate Employees

In [ ]:
# Keep only first occurrence of each employee_id
employees = employees.drop_duplicates(subset='employee_id', keep='first')

# Reset the row index numbers
employees = employees.reset_index(drop=True)

print('After removing duplicates:')
print(employees)

## 7. Clean Employee Names (Remove Extra Spaces)

In [ ]:
# .str.strip() is like str_trim() in R
employees['name'] = employees['name'].str.strip()

print('Clean names:')
print(employees['name'].tolist())

## 8. Standardize Gender

In [ ]:
# Map all variations to 'Male' or 'Female'
male_values   = ['M', 'Male', 'male', 'MALE']
female_values = ['F', 'Female', 'female', 'FEMALE']

def standardize_gender(value):
    if value in male_values:
        return 'Male'
    elif value in female_values:
        return 'Female'
    else:
        return np.nan

employees['gender'] = employees['gender'].apply(standardize_gender)

print('Unique gender values after cleaning:')
print(employees['gender'].unique())

print('\nCount by gender:')
print(employees['gender'].value_counts())

## 9. Standardize Department Names

In [ ]:
# Step 1: Remove spaces
employees['department'] = employees['department'].str.strip()

# Step 2: Map to standard names using lowercase comparison
def standardize_dept(value):
    v = value.lower()
    if v == 'it':
        return 'IT'
    elif v == 'hr':
        return 'HR'
    elif v == 'finance':
        return 'Finance'
    else:
        return value

employees['department'] = employees['department'].apply(standardize_dept)

print('Unique departments after cleaning:')
print(employees['department'].unique())

## 10. Convert Salary from String to Number

In [ ]:
print('Before cleaning:')
print(employees['salary'])

# Remove '$' and ',' then convert to float
employees['salary'] = (
    employees['salary']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

print('\nAfter cleaning:')
print(employees['salary'])
print('\nData type:', employees['salary'].dtype)

## 11. Handle Missing Age (Fill with Median)

In [ ]:
age_median = employees['age'].median()
print('Median age:', age_median)

# Fill missing age with median
employees['age'] = employees['age'].fillna(age_median)

print('\nAges after filling:')
print(employees['age'].tolist())

## 12. Handle Missing Salary (Fill with Department Median)

In [ ]:
# Fill missing salary using each department's own median
employees['salary'] = employees.groupby('department')['salary'].transform(
    lambda x: x.fillna(x.median())
)

# Check: any salary still missing?
print('Missing salaries after fill:', employees['salary'].isnull().sum())
print(employees[['name', 'department', 'salary']])

## 13. Create Monthly Salary

In [ ]:
employees['monthly_salary'] = employees['salary'] / 12

print(employees[['employee_id', 'name', 'salary', 'monthly_salary']])

## 14. Create Salary Categories

In [ ]:
# Like case_when() in R
def salary_category(salary):
    if salary < 50000:
        return 'Low'
    elif salary < 70000:
        return 'Medium'
    else:
        return 'High'

employees['salary_category'] = employees['salary'].apply(salary_category)

print('Salary category counts:')
print(employees['salary_category'].value_counts())

## 15. Create Age Groups

In [ ]:
def age_group(age):
    if age < 30:
        return 'Young'
    elif age < 40:
        return 'Mid-career'
    else:
        return 'Senior'

employees['age_group'] = employees['age'].apply(age_group)

print('Age group counts:')
print(employees['age_group'].value_counts())

## 16. Select Columns

In [ ]:
# Select specific columns
print('--- Selected columns ---')
print(employees[['employee_id', 'name', 'age', 'gender', 'salary']])

# Drop a column
print('\n--- All columns except gender ---')
print(employees.drop(columns=['gender']))

# Columns that start with 'salary'
salary_cols = [col for col in employees.columns if col.startswith('salary')]
print('\n--- Salary columns ---')
print(employees[salary_cols])

## 17. Filter Observations

In [ ]:
print('--- Employees older than 30 ---')
print(employees[employees['age'] > 30])

print('\n--- Salary above 60,000 ---')
print(employees[employees['salary'] > 60000])

print('\n--- IT department only ---')
print(employees[employees['department'] == 'IT'])

print('\n--- HR with salary > 50,000 ---')
print(employees[(employees['department'] == 'HR') & (employees['salary'] > 50000)])

print('\n--- IT or HR ---')
print(employees[employees['department'].isin(['IT', 'HR'])])

## 18. Sort Observations

In [ ]:
print('--- Lowest salary first ---')
print(employees.sort_values('salary'))

print('\n--- Highest salary first ---')
print(employees.sort_values('salary', ascending=False))

print('\n--- By department, then salary (desc) ---')
print(employees.sort_values(['department', 'salary'], ascending=[True, False]))

## 19. Count Employees

In [ ]:
print('Total employees:', len(employees))

print('\n--- By department ---')
print(employees['department'].value_counts())

print('\n--- By gender ---')
print(employees['gender'].value_counts())

print('\n--- By department AND gender ---')
print(employees.groupby(['department', 'gender']).size().reset_index(name='count'))

## 20. Calculate Summary Statistics

In [ ]:
print('Average salary:', employees['salary'].mean())

print('\n--- Multiple stats ---')
stats = {
    'employees'     : len(employees),
    'average_salary': employees['salary'].mean(),
    'median_salary' : employees['salary'].median(),
    'min_salary'    : employees['salary'].min(),
    'max_salary'    : employees['salary'].max(),
    'total_salary'  : employees['salary'].sum()
}
for key, val in stats.items():
    print(f'  {key}: {val}')

## 21. Salary by Department

In [ ]:
department_summary = employees.groupby('department').agg(
    employees      = ('salary', 'count'),
    average_salary = ('salary', 'mean'),
    median_salary  = ('salary', 'median'),
    min_salary     = ('salary', 'min'),
    max_salary     = ('salary', 'max'),
    total_salary   = ('salary', 'sum')
).reset_index().sort_values('average_salary', ascending=False)

print(department_summary)

## 22. Salary by Gender

In [ ]:
gender_summary = employees.groupby('gender').agg(
    employees      = ('salary', 'count'),
    average_salary = ('salary', 'mean'),
    median_salary  = ('salary', 'median')
).reset_index()

print(gender_summary)

## 23. Department + Gender Analysis

In [ ]:
dept_gender = employees.groupby(['department', 'gender']).agg(
    employees      = ('salary', 'count'),
    average_salary = ('salary', 'mean')
).reset_index()

print(dept_gender)

## 24. Build a Clean Pipeline

In [ ]:
# Like the pipe (%>%) chain in R
employees_clean = (
    employees
    [employees['age'] >= 18]                         # filter
    .assign(monthly_salary=lambda df: df['salary'] / 12)  # mutate
    .sort_values('salary', ascending=False)           # arrange desc
    .reset_index(drop=True)
)

print(employees_clean)

## 25. Joining Two Datasets — Create Benefits Table

In [ ]:
benefits = pd.DataFrame({
    'employee_id': [101, 102, 103, 104, 105, 106],
    'bonus'      : [5000, 4000, 6000, 3000, 8000, 5000],
    'insurance'  : ['Yes', 'Yes', 'Yes', 'No', 'Yes', 'Yes']
})

print(benefits)

## 26. Left Join

In [ ]:
# All employees kept; bonus/insurance filled with NaN if not found
employee_full = employees.merge(benefits, on='employee_id', how='left')
print(employee_full)

## 27. Inner Join

In [ ]:
# Only employees present in BOTH tables
inner = employees.merge(benefits, on='employee_id', how='inner')
print(inner)

## 28. Full Join

In [ ]:
# All rows from BOTH tables
full = employees.merge(benefits, on='employee_id', how='outer')
print(full)

## 29. Calculate Total Compensation

In [ ]:
# Replace NaN bonus with 0 before adding
employee_full['total_compensation'] = (
    employee_full['salary'] + employee_full['bonus'].fillna(0)
)

print(employee_full[['employee_id', 'name', 'salary', 'bonus', 'total_compensation']])

## 30. Reshape Data — Wide to Long (pivot_longer)

In [ ]:
# Wide format student marks
student_marks = pd.DataFrame({
    'student': ['Ali', 'Sara', 'Ahmed'],
    'Math'   : [80, 90, 75],
    'English': [75, 85, 80],
    'Science': [90, 88, 78]
})

print('Wide format:')
print(student_marks)

# Convert to long format — like pivot_longer() in R
marks_long = student_marks.melt(
    id_vars   ='student',
    var_name  ='subject',
    value_name='score'
)

print('\nLong format:')
print(marks_long)

## 31. Long to Wide (pivot_wider)

In [ ]:
marks_wide = marks_long.pivot(index='student', columns='subject', values='score').reset_index()
marks_wide.columns.name = None  # remove the 'subject' label from column axis

print('Back to wide format:')
print(marks_wide)

## 32. Data Validation

In [ ]:
print('--- Missing values per column ---')
print(employees.isnull().sum())

print('\n--- Duplicate employee IDs ---')
dup = employees['employee_id'].value_counts()
print(dup[dup > 1])

print('\n--- Impossible ages (< 18 or > 100) ---')
print(employees[(employees['age'] < 18) | (employees['age'] > 100)])

print('\n--- Salary range ---')
print('Min:', employees['salary'].min())
print('Max:', employees['salary'].max())
print('Avg:', employees['salary'].mean())

print('\n--- Unique gender values ---')
print(employees['gender'].unique())

print('\n--- Unique department values ---')
print(employees['department'].unique())

print('\n--- Unique salary categories ---')
print(employees['salary_category'].unique())

## 33. Scikit-Learn — Label Encoding

In [ ]:
# scikit-learn: encode categorical columns as numbers (useful for ML models)
le = LabelEncoder()

employees['gender_encoded']     = le.fit_transform(employees['gender'])
employees['department_encoded'] = le.fit_transform(employees['department'])

print(employees[['name', 'gender', 'gender_encoded', 'department', 'department_encoded']])

## 34. Matplotlib — Bar Chart: Salary by Department

In [ ]:
dept_avg = employees.groupby('department')['salary'].mean().sort_values(ascending=False)

plt.figure(figsize=(7, 4))
plt.bar(dept_avg.index, dept_avg.values, color=['steelblue', 'coral', 'mediumseagreen'])
plt.title('Average Salary by Department')
plt.xlabel('Department')
plt.ylabel('Average Salary')
plt.tight_layout()
plt.show()

## 35. Seaborn — Box Plot: Salary Distribution by Department

In [ ]:
plt.figure(figsize=(7, 4))
sns.boxplot(data=employees, x='department', y='salary', palette='Set2')
plt.title('Salary Distribution by Department')
plt.xlabel('Department')
plt.ylabel('Salary')
plt.tight_layout()
plt.show()

## 36. Seaborn — Bar Chart: Count by Gender

In [ ]:
plt.figure(figsize=(5, 4))
sns.countplot(data=employees, x='gender', palette='pastel')
plt.title('Number of Employees by Gender')
plt.xlabel('Gender')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 37. Seaborn — Scatter Plot: Age vs Salary

In [ ]:
plt.figure(figsize=(7, 4))
sns.scatterplot(data=employees, x='age', y='salary', hue='department', s=100)
plt.title('Age vs Salary (coloured by Department)')
plt.xlabel('Age')
plt.ylabel('Salary')
plt.tight_layout()
plt.show()

## 38. Seaborn — Heatmap: Salary Category vs Department

In [ ]:
pivot = employees.pivot_table(
    index='department',
    columns='salary_category',
    values='employee_id',
    aggfunc='count',
    fill_value=0
)

plt.figure(figsize=(6, 4))
sns.heatmap(pivot, annot=True, fmt='d', cmap='YlOrRd')
plt.title('Employee Count: Department vs Salary Category')
plt.tight_layout()
plt.show()

## 39. Matplotlib — Histogram: Age Distribution

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(employees['age'], bins=5, color='slateblue', edgecolor='white')
plt.title('Age Distribution of Employees')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## 40. Final Dataset

In [ ]:
final = employees[[
    'employee_id', 'name', 'age', 'gender',
    'department', 'salary', 'monthly_salary',
    'salary_category', 'age_group'
]].copy()

print('=== FINAL CLEANED DATASET ===')
print(final.to_string(index=False))

print('\n=== DEPARTMENT SUMMARY ===')
print(department_summary.to_string(index=False))

print('\n=== GENDER SUMMARY ===')
print(gender_summary.to_string(index=False))